# Real-Time Crypto Market Data

This notebook streams real-time market data from **Binance WebSocket API** and visualizes it.

- Live price ticker for BTC/USDT, ETH/USDT
- Candlestick (OHLCV) charts
- Price correlation between pairs

**No API key required** - Binance public WebSocket is completely free.

In [2]:
import json
import asyncio
import datetime
import requests

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import websockets

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Fetch Recent Klines (REST API)

First, let's pull historical candlestick data via Binance REST API to have a baseline chart.

In [3]:
def fetch_klines(symbol: str, interval: str = '1h', limit: int = 100) -> pd.DataFrame:
    """Fetch kline/candlestick data from Binance REST API (no auth required)."""
    url = 'https://api.binance.com/api/v3/klines'
    params = {'symbol': symbol, 'interval': interval, 'limit': limit}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    
    cols = [
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades', 'taker_buy_base',
        'taker_buy_quote', 'ignore'
    ]
    df = pd.DataFrame(resp.json(), columns=cols)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')
    for c in ['open', 'high', 'low', 'close', 'volume', 'quote_volume']:
        df[c] = df[c].astype(float)
    return df

In [4]:
btc = fetch_klines('BTCUSDT', '1h', 200)
eth = fetch_klines('ETHUSDT', '1h', 200)

print(f"BTC/USDT: {len(btc)} candles, latest close: ${btc['close'].iloc[-1]:,.2f}")
print(f"ETH/USDT: {len(eth)} candles, latest close: ${eth['close'].iloc[-1]:,.2f}")
btc.tail(3)

BTC/USDT: 200 candles, latest close: $74,702.72
ETH/USDT: 200 candles, latest close: $2,354.80


,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
197,2026-03-16 21:00:00,74236.90,74799.00,74150.00,74401.36,1246.95221,2026-03-16 21:59:59.999,9.285101e+07,217232,693.47258000,51648436.99290530,0
198,2026-03-16 22:00:00,74401.35,74899.99,74378.43,74772.66,1058.73068,2026-03-16 22:59:59.999,7.914179e+07,202153,625.36327000,46748104.64548710,0
199,2026-03-16 23:00:00,74773.56,74773.56,74250.21,74702.72,772.88138,2026-03-16 23:59:59.999,5.758418e+07,110564,421.77595000,31429471.19787280,0


## 2. Candlestick Chart with Volume

In [5]:
def plot_candlestick(df: pd.DataFrame, title: str):
    """Interactive candlestick chart with volume bars."""
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        vertical_spacing=0.03,
        row_heights=[0.7, 0.3]
    )
    fig.add_trace(
        go.Candlestick(
            x=df['open_time'], open=df['open'], high=df['high'],
            low=df['low'], close=df['close'], name='OHLC'
        ), row=1, col=1
    )
    colors = ['red' if r['close'] < r['open'] else 'green' for _, r in df.iterrows()]
    fig.add_trace(
        go.Bar(x=df['open_time'], y=df['volume'], marker_color=colors, name='Volume'),
        row=2, col=1
    )
    fig.update_layout(
        title=title, height=600, xaxis_rangeslider_visible=False,
        template='plotly_dark'
    )
    fig.show()

plot_candlestick(btc, 'BTC/USDT - 1H Candles')

In [6]:
plot_candlestick(eth, 'ETH/USDT - 1H Candles')

## 3. BTC-ETH Price Correlation (Pairs Trading Analysis)

This is directly relevant to the pairs trading strategy from our Rust engine.

In [ ]:
merged = pd.DataFrame({
    'time': btc['open_time'],
    'btc_close': btc['close'].values,
    'eth_close': eth['close'].values
})

merged['btc_return'] = merged['btc_close'].pct_change()
merged['eth_return'] = merged['eth_close'].pct_change()
merged['spread'] = merged['btc_close'] / merged['eth_close']

lookback = 60
merged['spread_mean'] = merged['spread'].rolling(lookback).mean()
merged['spread_std'] = merged['spread'].rolling(lookback).std()
merged['z_score'] = (merged['spread'] - merged['spread_mean']) / merged['spread_std']

corr = merged['btc_return'].corr(merged['eth_return'])
print(f"BTC-ETH return correlation: {corr:.4f}")
merged.dropna().tail(5)

In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.4, 0.3, 0.3],
    subplot_titles=('BTC vs ETH (Normalized)', 'BTC/ETH Spread Ratio', 'Z-Score (Pairs Signal)')
)

# Normalized prices
fig.add_trace(
    go.Scatter(x=merged['time'], y=merged['btc_close'] / merged['btc_close'].iloc[0],
               name='BTC (norm)', line=dict(color='orange')), row=1, col=1
)
fig.add_trace(
    go.Scatter(x=merged['time'], y=merged['eth_close'] / merged['eth_close'].iloc[0],
               name='ETH (norm)', line=dict(color='cyan')), row=1, col=1
)

# Spread ratio with mean
fig.add_trace(
    go.Scatter(x=merged['time'], y=merged['spread'], name='Spread', line=dict(color='white')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=merged['time'], y=merged['spread_mean'], name='Mean',
               line=dict(color='yellow', dash='dash')), row=2, col=1
)

# Z-score with entry/exit bands
fig.add_trace(
    go.Scatter(x=merged['time'], y=merged['z_score'], name='Z-Score',
               line=dict(color='lime')), row=3, col=1
)
for level, color in [(2.0, 'red'), (-2.0, 'red'), (0.5, 'gray'), (-0.5, 'gray')]:
    fig.add_hline(y=level, line_dash='dot', line_color=color, row=3, col=1)

fig.update_layout(
    height=800, template='plotly_dark',
    title=f'BTC-ETH Pairs Trading Analysis (Correlation: {corr:.4f})'
)
fig.show()

## 4. Real-Time WebSocket Streaming

Stream live trades from Binance WebSocket. Run the cell and watch prices update in real time.

**Interrupt the kernel to stop streaming.**

In [ ]:
async def stream_trades(symbols: list[str], duration_sec: int = 30):
    """Stream real-time trades from Binance WebSocket."""
    streams = '/'.join(f"{s.lower()}@trade" for s in symbols)
    url = f"wss://stream.binance.com:9443/ws/{streams}"
    
    trades = []
    start = datetime.datetime.now()
    
    async with websockets.connect(url) as ws:
        print(f"Connected to Binance WebSocket. Streaming for {duration_sec}s...")
        while (datetime.datetime.now() - start).seconds < duration_sec:
            msg = json.loads(await ws.recv())
            trade = {
                'symbol': msg['s'],
                'price': float(msg['p']),
                'qty': float(msg['q']),
                'time': pd.to_datetime(msg['T'], unit='ms'),
                'buyer_maker': msg['m']
            }
            trades.append(trade)
            
            if len(trades) % 50 == 0:
                print(f"  [{trade['time'].strftime('%H:%M:%S')}] {trade['symbol']}: "
                      f"${trade['price']:,.2f}  (qty: {trade['qty']:.4f})  "
                      f"[{len(trades)} trades collected]")
    
    print(f"\nDone! Collected {len(trades)} trades.")
    return pd.DataFrame(trades)

# Stream BTC and ETH trades for 30 seconds
live_trades = await stream_trades(['BTCUSDT', 'ETHUSDT'], duration_sec=30)

In [ ]:
# Visualize the collected live trades
for symbol in live_trades['symbol'].unique():
    df_sym = live_trades[live_trades['symbol'] == symbol]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_sym['time'], y=df_sym['price'],
        mode='lines', name=symbol,
        line=dict(width=1)
    ))
    fig.update_layout(
        title=f'{symbol} Live Trades ({len(df_sym)} trades)',
        template='plotly_dark', height=400,
        xaxis_title='Time', yaxis_title='Price (USDT)'
    )
    fig.show()

print(f"\nTrade counts: {live_trades['symbol'].value_counts().to_dict()}")
print(f"BTC price range: ${live_trades[live_trades.symbol=='BTCUSDT']['price'].min():,.2f}"
      f" - ${live_trades[live_trades.symbol=='BTCUSDT']['price'].max():,.2f}")

## 5. Available Binance WebSocket Streams

Reference for building more advanced streaming:

| Stream | URL suffix | Description |
|--------|-----------|-------------|
| Trade | `<symbol>@trade` | Individual trades |
| Kline | `<symbol>@kline_<interval>` | Candlestick updates (1m, 5m, 1h...) |
| Ticker | `<symbol>@ticker` | 24hr rolling stats |
| Mini Ticker | `<symbol>@miniTicker` | Lightweight price info |
| Book Ticker | `<symbol>@bookTicker` | Best bid/ask |
| Depth | `<symbol>@depth<levels>` | Order book (5, 10, 20 levels) |
| Agg Trade | `<symbol>@aggTrade` | Aggregated trades |

All streams are **free** and require **no authentication**.

Base URL: `wss://stream.binance.com:9443/ws/`